In [37]:
from typing import List
import stim
from stimcirq import stim_circuit_to_cirq_circuit
import cirq
import openfermion as of
from encoded.code_extension import encoding_unitary_for_new_stabilizer
from encoded.utils import cirq_pauli_string_to_stim

In [38]:
n = 5
def extended_repetition_generators(n: int) -> List[stim.PauliString]:
    generators = []
    for i in range(n-1):
        pauli_str = 'I' * i + 'Z' * 2 + 'I' * (n - i - 2)
        assert len(pauli_str) == n
        generators.append(stim.PauliString(pauli_str))
    generators.append(stim.PauliString('X' * n))
    return generators

In [39]:
generators = extended_repetition_generators(n)
for generator in generators:
    print(generator)

+ZZ___
+_ZZ__
+__ZZ_
+___ZZ
+XXXXX


In [40]:
for gi in generators:
    for gj in generators:
        assert gi.commutes(gj)

In [41]:
def all_single_qubit_errs(n: int) -> List[stim.PauliString]:
    """Returns the set of all single-qubit Pauli errors."""

    errs = []
    for i in range(n):
        for p in ['X', 'Y', 'Z']:
            pauli_str = "I" * i + p + 'I' * (n - i - 1)
            assert len(pauli_str) == n
            errs.append(stim.PauliString(pauli_str))
    return errs

In [42]:
n = 5
errors = all_single_qubit_errs(n)

In [43]:
number_false = 0
number_checked = 0
for i, ei in enumerate(errors):
    for j in range(i):
        number_checked += 1
        ej = errors[j]
        e = ei * ej
        commutators = []
        for generator in generators:
            comm = e.commutes(generator)
            commutators.append(comm)
        has_anticommuting_operator = any([not b for b in commutators])
        if has_anticommuting_operator:
            number_false += 1
        if not has_anticommuting_operator:
            print(f"{ei} * {ej} = {e}, {commutators} {has_anticommuting_operator} ")
print(f"{number_false}/{number_checked} operators anticommute.")

+_Z___ * +Z____ = +ZZ___, [True, True, True, True, True] False 
+__Z__ * +Z____ = +Z_Z__, [True, True, True, True, True] False 
+__Z__ * +_Z___ = +_ZZ__, [True, True, True, True, True] False 
+___Z_ * +Z____ = +Z__Z_, [True, True, True, True, True] False 
+___Z_ * +_Z___ = +_Z_Z_, [True, True, True, True, True] False 
+___Z_ * +__Z__ = +__ZZ_, [True, True, True, True, True] False 
+____Z * +Z____ = +Z___Z, [True, True, True, True, True] False 
+____Z * +_Z___ = +_Z__Z, [True, True, True, True, True] False 
+____Z * +__Z__ = +__Z_Z, [True, True, True, True, True] False 
+____Z * +___Z_ = +___ZZ, [True, True, True, True, True] False 
95/105 operators anticommute.
